# Treinamento de Modelos: MLflow
## O "Dataset" Iris com MLflow "Autolog"

Antes de usarmos ferramentas de automação, precisamos entender como o **MLflow** intercepta nosso treinamento tradicional para garantir a reprodutibilidade. O Unity Catalog agora atua nativamente como o nosso **Model Registry** (Repositório de Modelos).

In [0]:
%%capture
%pip install databricks
# %pip install databricks-automl --upgrade
dbutils.library.restartPython()

## Vamos recriar a tabela do dataset iris com a coluna `id_flor`

In [0]:
# Catalogo e Esquema do dataset Iris
catalogo_origem = "workspace"
esquema_origem = "default"
tabela_iris_raw = f"{catalogo_origem}.{esquema_origem}.iris_dataset"

# Lendo os dados brutos que já estão no ambiente
df_iris = spark.table(tabela_iris_raw)

# Garanta que a tabela possua a chave primária (id_flor) e as colunas ajustadas.

# Registra o DataFrame atual como uma View temporária no Spark
df_iris.createOrReplaceTempView("vw_iris_raw")

# Executa a query SQL para construir a chave primária
df_iris_sql = spark.sql("""
    SELECT 
        CONCAT('flor_', ROW_NUMBER() OVER (ORDER BY (SELECT NULL))) AS id_flor,
        sepal_length,
        sepal_width,
        petal_length,
        petal_width,
        species 
    FROM vw_iris_raw
""")